# 04 · NE Brazil training and experiments (P4, E2, E3, E4)
Trains on the notebook 02 data (26 INMET stations at control ratio 0.6, ERA5 `NEBrazil`) and runs the analyses of
`PLAN_UPDATE_v2.md`. Results go to `docs/results_brazil.md`; LaTeX tables to `$DATA_ROOT/tables/`.

All metrics use real observations only, with 95% day-bootstrap intervals. With lead 1 h and ERA5 at the target hour,
the results are **reanalysis downscaling**, like the paper's ERA5 / HRRR-A results (only HRRR-F is a forecast).

Runtime: GPU. 11 trainings of 50 epochs on 26 stations. Every training is skipped when its test outputs exist, so after
a Colab disconnect just run all cells again. Sections are in priority order.

In [ ]:
# Setup. Colab: clones/updates the repo, installs requirements, mounts Drive.
# PC: start Jupyter anywhere inside the repo and set the DATA_ROOT environment variable first.
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo = Path('/content/LocalizedWeatherBR')
    if repo.exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', 'https://github.com/joancsz/LocalizedWeatherBR.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/LocalizedWeather'
else:
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PLAN_BRAZIL.md').exists()))
    DATA_ROOT = os.environ.get('DATA_ROOT') or sys.exit('set DATA_ROOT (folder holding neus/ and brazil/) before starting Jupyter')

os.environ['DATA_ROOT'] = DATA_ROOT  # configs read ${DATA_ROOT}
PY = sys.executable                   # !{PY} runs the scripts with this kernel's Python on Colab and Windows alike
print('repo:', os.getcwd(), '| DATA_ROOT:', DATA_ROOT)

In [ ]:
!{PY} -m pytest -q tests/

In [ ]:
import torch, yaml
assert torch.cuda.is_available(), 'switch to a GPU runtime'
print(torch.cuda.get_device_name(0))

OUT = Path(DATA_ROOT) / 'brazil' / 'ModelOutputs'


def train(name, base='configs/ne_brazil_vit.yaml', **extra):
    """Train config base + extra into ModelOutputs/<name>, unless its test predictions exist."""
    run_dir = OUT / name
    if (run_dir / 'Preds_CUSTOM_min.pkl').exists():
        print('skip', name)
        return run_dir
    cfg = {**yaml.safe_load(open(base)), **extra, 'output_saving_path': f'ModelOutputs/{name}'}
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = run_dir / 'config.yaml'
    yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
    print('=== train', name, extra, flush=True)
    !{PY} Source/run.py --config "{cfg_path}"
    if _exit_code:
        raise RuntimeError(f'{name} failed (exit {_exit_code})')
    return run_dir

## P4 / E2 · Transformer + ERA5, baselines
The Transformer run also builds the ERA5-nearest file that every baseline reads.

In [ ]:
VIT = train('ne_brazil_vit')
!{PY} scripts/eval/baselines.py --run-dir "{VIT}"

## E3 · Sea breeze: error by local hour, coastal (< 20 km) vs inland
The log lists how many stations fall within 10/20/30/50 km of the coast.

In [ ]:
from IPython.display import Image, display
!{PY} scripts/eval/diurnal_analysis.py --run-dir "{VIT}" --coast-km 20 --utc-offset -3
display(Image(VIT / 'diurnal' / 'wind_error.png'), Image(VIT / 'diurnal' / 'uv_cycle.png'))

## P4 / E2 · GNN + ERA5 grid nodes

In [ ]:
GNN = train('ne_brazil_gnn', base='configs/ne_brazil_gnn.yaml')
!{PY} scripts/eval/baselines.py --run-dir "{GNN}"
!{PY} scripts/eval/diurnal_analysis.py --run-dir "{GNN}" --coast-km 20 --utc-offset -3

## E4 · Station density
Keep 75 / 50 / 25 % of the 26 stations (seeds 0–2); fraction 1.0 is the Transformer run above. Every network keeps the
same 6 spread-out evaluation stations, and every run is scored on those 6 only.
At 25 % the network *is* the 6 evaluation stations (round(0.25 × 26) = 6), so the seeds would train identical networks:
only seed 0 runs there.

In [ ]:
DENSITY = [(f, s) for f in (0.75, 0.5) for s in (0, 1, 2)] + [(0.25, 0)]
for f, s in DENSITY:
    train(f'ne_brazil_e4_frac{f}_seed{s}', station_keep_fraction=f, station_subsample_seed=s)
for run_dir in [VIT] + [OUT / f'ne_brazil_e4_frac{f}_seed{s}' for f, s in DENSITY]:
    !{PY} scripts/eval/station_density.py --run-dir "{run_dir}" --eval-set fixed

## E4 · Availability filter
Control ratio 0.7 and 0.8 (0.6 is the Transformer run above). Each stricter filter keeps a subset of the looser one's
stations, so all three are scored on the stations kept at 0.8. The station count of each run is in its table.

In [ ]:
FILTER = (0.7, 0.8)
for r in FILTER:
    train(f'ne_brazil_e4_ratio{r}', madis_control_ratio=r)
for run_dir in [VIT] + [OUT / f'ne_brazil_e4_ratio{r}' for r in FILTER]:
    !{PY} scripts/eval/station_density.py --run-dir "{run_dir}" --eval-set ratio:0.8

## Tables
Also picks up the degraded NE-US runs of notebook 03 that have finished.

In [ ]:
!{PY} scripts/eval/make_tables.py --data-root "{DATA_ROOT}"
for f in sorted((Path(DATA_ROOT) / 'tables').glob('*.tex')):
    print(f'%% {f.name}\n' + f.read_text())